# 02 · El Niño (ONI)

Índice ONI da NOAA e suas features. Cada trimestre vai para o seu último mês (DJF → fevereiro), para não usar dado do futuro.

In [ ]:
import sys
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, dados, features

ETAPA = "02_enso"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

## 1. ONI

Acima de 0,5 é El Niño; abaixo de −0,5 é La Niña.

In [ ]:
oni = dados.baixar_oni().loc[config.INICIO:]
print(f"{len(oni)} meses, de {oni.index[0]} a {oni.index[-1]}")

ax = oni.plot(color="tab:red", title="ONI")
ax.axhline(.5, ls="--", c="gray", lw=.7); ax.axhline(-.5, ls="--", c="gray", lw=.7)
ax.fill_between(oni.index.to_timestamp(), oni, 0, where=oni > .5, color="tab:red", alpha=.3)
ax.fill_between(oni.index.to_timestamp(), oni, 0, where=oni < -.5, color="tab:blue", alpha=.3)
plt.gcf().savefig(PASTA / "oni.png", dpi=120); plt.show()
oni.tail()

## 2. Com quanto atraso o ONI se relaciona com o preço

Correlação do ONI de *k* meses antes com o alvo, só nos anos antes do teste.

In [ ]:
preco = pd.read_csv(config.ler_resultado("01_preco", "preco.csv"), index_col="mes")
preco.index = pd.PeriodIndex(preco.index, freq="M")
treino = preco.loc[:pd.Period(config.TESTE_INICIO) - 1, "alvo"]

atrasos = range(0, 7)
corr = pd.Series([treino.corr(oni.shift(k).reindex(treino.index)) for k in atrasos], index=atrasos)
corr.plot.bar(color="tab:red", rot=0, title="Correlação do ONI de k meses antes com o alvo")
plt.axhline(0, color="k", lw=.8); plt.xlabel("k (meses)"); plt.show()
corr.round(3)

## 3. Features

- `oni`: ONI do mês
- `oni_lag2`: ONI de 2 meses antes
- `oni_var3`: quanto o ONI mudou em 3 meses

In [ ]:
enso = features.features_enso(oni)
juntos = enso.join(preco["alvo"], how="inner").loc[:pd.Period(config.TESTE_INICIO) - 1]
juntos.drop(columns="alvo").corrwith(juntos["alvo"]).plot.bar(color="tab:red", rot=0, title="Correlação com o alvo (antes do teste)")
plt.axhline(0, color="k", lw=.8); plt.show()
enso.tail()

## 4. Salvando

In [ ]:
saida = enso.copy()
saida.index = saida.index.astype(str)
saida.index.name = "mes"
saida.to_csv(PASTA / "oni.csv")
config.salvar_config(PASTA, fonte="NOAA CPC oni.ascii.txt", meses=len(saida), ultimo_mes=saida.index[-1])
print(sorted(p.name for p in PASTA.iterdir()))